In [1]:
import pandas as pd
loans_raw = pd.read_csv('../data/loans.csv')

# 1)

In [2]:
loans = loans_raw.copy()

In [3]:
print(loans.return_date.isna().sum())

30


In [4]:
loans.return_date = pd.to_datetime(loans.return_date, errors='coerce')

In [5]:
print(loans.return_date.isna().sum())

30


In [6]:
loans.returned = loans.returned.str.lower().map({'yes': True, 'y': True, 'no': False, 'n': False})
print((loans.returned == False).sum())

30


Most likely they are missing return_date because they haven't been returned yet (makes sense)

In [7]:
print(loans.loc[loans.returned == False, 'return_date'].isna().sum())

30


Yes, the 30 that have returned == False are the same 30 that are missing a return_date.

In [8]:
loans.loan_date = pd.to_datetime(loans.loan_date, errors='coerce')

In [9]:
print((loans.loc[loans.return_date < loans.loan_date, 'loan_id']).count())

6


In [10]:
print((loans.loc[loans.return_date < loans.loan_date, ['loan_date', 'return_date']]))

     loan_date return_date
280 2026-03-24  2026-03-21
360 2026-04-16  2026-04-11
511 2026-05-28  2026-05-23
518 2026-05-31  2026-05-28
567 2026-06-12  2026-06-09
618 2026-06-27  2026-06-25


Setting all return_dates that are before the loan_date to exactly the loan_date.

In [11]:
loans.loc[loans.return_date < loans.loan_date, 'return_date'] = loans.loan_date

In [12]:
print((loans.loc[loans.return_date < loans.loan_date, 'loan_id']).count())

0


In [13]:
print((loans.return_date - loans.loan_date).dt.days.min())

0.0


# 2)

In [14]:
loans.genre = (
    loans.genre
    .str.strip()
    .str.lower()
    .str.title()
    .replace("Children'S", "Children")
    .replace("Sci Fi", "Science Fiction")
    .replace("Sci-Fi", "Science Fiction")
)
loans.late_fee = (
    loans.late_fee
    .str.replace('kr', '')
    .str.replace(' ', '')
    .str.replace(',', '.')
).astype(float)
loans = loans.drop_duplicates().reset_index(drop=True)
loans = loans.drop(columns='notes')

In [15]:
print(loans.duplicated(subset=['member_id', 'title', 'loan_date'], keep=False).sum())

10


Looks like a clear mistake, let's remove the duplicates

In [16]:
previous_late_fees = loans.late_fee.sum()
print(previous_late_fees)

7447.5


In [17]:
loans = loans.drop_duplicates(subset=['member_id', 'title', 'loan_date']).reset_index(drop=True)

In [20]:
print(loans.late_fee.sum())

7362.5


In [21]:
print(previous_late_fees - loans.late_fee.sum())

85.0


In [18]:
loans.shape

(600, 8)

# 3)

In [25]:
def clean_loans(raw):
    loans = raw.copy()
    loans.genre = (
        loans.genre
        .str.strip()
        .str.lower()
        .str.title()
        .replace("Children'S", "Children")
        .replace("Sci Fi", "Science Fiction")
        .replace("Sci-Fi", "Science Fiction")
    )
    loans.returned = loans.returned.str.lower().map({'yes': True, 'y': True, 'no': False, 'n': False})
    loans.late_fee = (
        loans.late_fee
        .str.replace('kr', '')
        .str.replace(' ', '')
        .str.replace(',', '.')
    ).astype(float)
    loans.loan_date = pd.to_datetime(loans.loan_date, errors='coerce')
    loans.return_date = pd.to_datetime(loans.return_date, errors='coerce')
    loans = loans.drop(columns='notes')
    loans.loc[loans.return_date < loans.loan_date, 'return_date'] = loans.loan_date
    loans = loans.drop_duplicates(subset=['member_id', 'title', 'loan_date']).reset_index(drop=True)
    return loans

In [30]:
loans = clean_loans(loans_raw)
loans.shape

(600, 8)

In [31]:
loans.head()

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
0,L-30001,M0150,A Second Spring,Romance,2026-01-06,2026-01-30,7.5,True
1,L-30002,M0037,The Silent Harbour,Crime,2026-01-05,2026-01-25,0.0,True
2,L-30003,M0143,The Glass Mage,Fantasy,2026-01-05,2026-01-26,0.0,True
3,L-30004,M0121,Dead Reckoning Lane,Crime,2026-01-05,2026-01-12,0.0,True
4,L-30005,M0109,Ollie Learns to Swim,Children,2026-01-06,2026-02-02,15.0,True


In [32]:
print(loans.late_fee.sum())

7362.5


In [37]:
print(loans.return_date.isna().sum())

30


Reproducing the result of a bunch of cells exactly, requires restarting and running all the cells again. If we put it in a function, not only is it defined as long as we have run that one single cell, if we want to change something all we need to do is change the function and re-run the function cell.

# 4)

In [89]:
loans['has_late_fee'] = loans['late_fee'] > 0

In [93]:
loans.groupby('genre').agg(
    n_loans = ('loan_id', 'count'),
    fee_rate = ('has_late_fee', 'mean'),
    avg_fee = ('late_fee', 'mean')
).sort_values('fee_rate')

,n_loans,fee_rate,avg_fee
genre,,,
Fantasy,87,0.448276,12.356322
Crime,155,0.464516,11.209677
Children,130,0.469231,10.903846
History,57,0.508772,13.333333
Romance,82,0.512195,14.786585
Science Fiction,89,0.539326,13.033708


Science Fiction has the highest share of late loans, Romance has the highest average late fee.
Differance between the genres doesn't seem very large, so it's hard to draw any significant conclusions.

# 5)

Out of all loans without a late fee and with a return date, what's the longest duration of such a loan?

In [104]:
returned_not_late = loans.where(loans.has_late_fee == False).where(loans.returned)
print((returned_not_late.return_date - returned_not_late.loan_date).dt.days.max())

21.0


In [107]:
print((loans.loc[(loans.return_date - loans.loan_date).dt.days == 21, ['loan_id', 'loan_date', 'return_date', 'late_fee']]))

     loan_id  loan_date return_date  late_fee
2    L-30003 2026-01-05  2026-01-26       0.0
12   L-30013 2026-01-09  2026-01-30       0.0
16   L-30017 2026-01-09  2026-01-30       0.0
20   L-30021 2026-01-10  2026-01-31       0.0
30   L-30031 2026-01-13  2026-02-03       0.0
97   L-30098 2026-02-03  2026-02-24       0.0
136  L-30137 2026-02-14  2026-03-07       0.0
157  L-30158 2026-02-19  2026-03-12       0.0
184  L-30185 2026-02-27  2026-03-20       0.0
237  L-30238 2026-03-15  2026-04-05       0.0
296  L-30297 2026-04-01  2026-04-22       0.0
328  L-30329 2026-04-10  2026-05-01       0.0
457  L-30458 2026-05-17  2026-06-07       0.0
468  L-30469 2026-05-20  2026-06-10       0.0
515  L-30516 2026-06-03  2026-06-24       0.0
558  L-30559 2026-06-16  2026-07-07       0.0
581  L-30582 2026-06-22  2026-07-13       0.0
590  L-30591 2026-06-26  2026-07-17       0.0


In [108]:
print((loans.loc[(loans.return_date - loans.loan_date).dt.days == 22, ['loan_id', 'loan_date', 'return_date', 'late_fee']]))

     loan_id  loan_date return_date  late_fee
11   L-30012 2026-01-09  2026-01-31       2.5
89   L-30090 2026-01-30  2026-02-21       2.5
146  L-30147 2026-02-16  2026-03-10       2.5
154  L-30155 2026-02-18  2026-03-12       2.5
163  L-30164 2026-02-21  2026-03-15       2.5
187  L-30188 2026-02-28  2026-03-22       2.5
210  L-30211 2026-03-06  2026-03-28       2.5
222  L-30223 2026-03-11  2026-04-02       2.5
229  L-30230 2026-03-12  2026-04-03       2.5
273  L-30274 2026-03-26  2026-04-17       2.5
308  L-30309 2026-04-04  2026-04-26       2.5
466  L-30467 2026-05-21  2026-06-12       2.5
482  L-30483 2026-05-25  2026-06-16       2.5
573  L-30574 2026-06-21  2026-07-13       2.5
585  L-30586 2026-06-24  2026-07-16       2.5


We can conclude that every book that was returned after 21 days was not charged with a late_fee and we can conclude that each book that was returned after 22 days was charged with a late fee of 2.5
We cannot conclude that every book that was returned after less than 21 days also wasn't charged with a late_fee (we didn't check yet). Nor can we conclude that every book that was returned after 23+ days was charged with a late_fee (we didn't check that either).